In [1]:
import os

In [2]:
%pwd

'/home/sharad/Projects/end_to_end_ml_project_with_mlflow/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/sharad/Projects/end_to_end_ml_project_with_mlflow'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir : Path
    source_URL : str
    local_data_file : Path
    unzip_dir : Path
    

In [6]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml , create_directories


In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.parmas = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)


        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])


        data_ingestion_config = DataIngestionConfig(
            root_dir= config.root_dir,
            source_URL= config.source_URL,
            local_data_file= config.local_data_file,
            unzip_dir=config.unzip_dir
        )


        return data_ingestion_config

In [8]:
import os
import urllib.request as request
import zipfile
from mlproject import logger
from mlproject.utils.common import get_size

In [ ]:
class DataIngestion:
    def __init__(self , config : DataIngestionConfig):
        self.config = config


    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename , headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )

            logger.info(f"{filename} download ! with following info : \n{headers}")
        else:
            logger.info(f"File already exists of size : {get_size(self.config.local_data_file)}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path , exist_ok=True)

        with zipfile.ZipFile(self.config.local_data_file , "r") as zip_ref:
            zip_ref.extractall(unzip_path)

    

    

In [11]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-11-06 20:13:39,542 : INFO : common : yaml file: config/config.yaml loaded successfully]
[2026-11-06 20:13:39,543 : INFO : common : yaml file: params.yaml loaded successfully]
[2026-11-06 20:13:39,544 : INFO : common : yaml file: schema.yaml loaded successfully]
[2026-11-06 20:13:39,544 : INFO : common : created directory at: artifacts]
[2026-11-06 20:13:39,545 : INFO : common : created directory at: artifacts/data_ingestion]
[2026-11-06 20:13:40,141 : INFO : 166774583 : artifacts/data_ingestion/data.zip download ! with following info : 
Connection: close
Content-Length: 91353
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "e09dc271395a544f6f655050dd47156058df5b4517da1bce195e95eb6e567889"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 4686:3EF8A6:827A8:AC83A:6AEDE69F
x-github-edge-reg